In [34]:
import pandas as pd
bakery = pd.read_csv("C:/Users/yurii/Downloads/IH_Projects/ml_project/bakery_sales.csv")
bakery.head()

,Unnamed: 0,date,time,ticket_number,article,Quantity,unit_price
0,0,2021-01-02,08:38,150040.0,BAGUETTE,1.0,"0,90 €"
1,1,2021-01-02,08:38,150040.0,PAIN AU CHOCOLAT,3.0,"1,20 €"
2,4,2021-01-02,09:14,150041.0,PAIN AU CHOCOLAT,2.0,"1,20 €"
3,5,2021-01-02,09:14,150041.0,PAIN,1.0,"1,15 €"
4,8,2021-01-02,09:25,150042.0,TRADITIONAL BAGUETTE,5.0,"1,20 €"


# Bakery Demand Forecasting

### Business Problem
The bakery needs to predict product demand to improve daily production planning and reduce waste.

### ML Objective
Predict next-day sales quantities for selected bakery products that generat 80% of revenue using historical sales data.

### Target Variable
`daily_quantity_sold` — total units sold per product per day.

**ML Task:** Supervised Learning — Regression.

In [35]:
# Dataset overview
print(bakery.shape)
bakery.info()

# Missing values
print(bakery.isnull().sum())

# Duplicate rows
print("Duplicates:", bakery.duplicated().sum())

(234005, 7)
<class 'pandas.DataFrame'>
RangeIndex: 234005 entries, 0 to 234004
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Unnamed: 0     234005 non-null  int64  
 1   date           234005 non-null  str    
 2   time           234005 non-null  str    
 3   ticket_number  234005 non-null  float64
 4   article        234005 non-null  str    
 5   Quantity       234005 non-null  float64
 6   unit_price     234005 non-null  str    
dtypes: float64(2), int64(1), str(4)
memory usage: 20.5 MB
Unnamed: 0       0
date             0
time             0
ticket_number    0
article          0
Quantity         0
unit_price       0
dtype: int64
Duplicates: 0


In [36]:
# Convert prices to numeric
bakery["unit_price"] = (
    bakery["unit_price"]
    .str.replace("€", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)

# Calculate revenue
bakery["revenue"] = bakery["Quantity"] * bakery["unit_price"]

# Revenue by product
product_revenue = (
    bakery.groupby("article")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

# Cumulative revenue share
product_revenue["cumulative_share"] = (
    product_revenue["revenue"].cumsum() / product_revenue["revenue"].sum()
)

# Compare 75% and 80% thresholds
for threshold in [0.75, 0.80]:
    n = (product_revenue["cumulative_share"] < threshold).sum() + 1
    print(f"{threshold:.0%} of revenue: {n} products")

product_revenue.head(30)

75% of revenue: 20 products
80% of revenue: 26 products


,article,revenue,cumulative_share
0,TRADITIONAL BAGUETTE,144756.05,0.259122
1,FORMULE SANDWICH,34710.50,0.321256
2,CROISSANT,33579.50,0.381365
3,PAIN AU CHOCOLAT,31100.45,0.437037
4,BANETTE,24555.15,0.480992
5,BAGUETTE,20481.40,0.517655
6,SANDWICH COMPLET,13550.00,0.541910
7,SPECIAL BREAD,13419.70,0.565932
8,TRAITEUR,12254.30,0.587868
9,GRAND FAR BRETON,10713.50,0.607046


### Product Selection — Pareto Analysis

To focus on the bakery's most commercially important products, we selected the top 26 products contributing approximately 80% of total revenue.

These products will be used for demand forecasting to support daily production planning.

In [37]:
product_revenue.head(26)[["article"]].to_csv(
    "selected_products.csv", index=False
)

### Data Cleaning
Remove unnecessary columns, correct data types, and check for anomalies.

In [38]:
# Remove unnecessary column
bakery = bakery.drop(columns=["Unnamed: 0"])

# Convert date to datetime
bakery["date"] = pd.to_datetime(bakery["date"])

In [39]:
# Load selected products
selected = pd.read_csv("selected_products.csv")

# Filter dataset
bakery = bakery[bakery["article"].isin(selected["article"])].copy()

# Check results
print("Rows:", len(bakery))
print("Products:", bakery["article"].nunique())

Rows: 173609
Products: 26


In [40]:
# Quantity distribution
print(bakery["Quantity"].describe())

# Check zero and negative quantities
print("Zero:", (bakery["Quantity"] == 0).sum())
print("Negative:", (bakery["Quantity"] < 0).sum())

count    173609.000000
mean          1.636626
std           1.233927
min         -18.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          55.000000
Name: Quantity, dtype: float64
Zero: 0
Negative: 980


In [41]:
# Inspect negative quantities
bakery[bakery["Quantity"] < 0].head(10)

,date,time,ticket_number,article,Quantity,unit_price,revenue
448,2021-01-03,10:41,150265.0,BAGUETTE,-1.0,0.90,-0.90
1162,2021-01-05,19:00,150738.0,SPECIAL BREAD,-1.0,2.40,-2.40
1468,2021-01-08,11:58,150941.0,BOULE 400G,-1.0,1.50,-1.50
1469,2021-01-08,11:58,150941.0,CROISSANT,-4.0,1.10,-4.40
1470,2021-01-08,11:58,150941.0,PAIN AU CHOCOLAT,-2.0,1.20,-2.40
1845,2021-01-09,13:23,151163.0,FORMULE SANDWICH,-1.0,6.50,-6.50
2043,2021-01-10,11:43,151280.0,VIK BREAD,-1.0,2.50,-2.50
2351,2021-01-11,12:31,151476.0,FORMULE SANDWICH,-1.0,6.50,-6.50
3445,2021-01-17,11:39,152169.0,CEREAL BAGUETTE,-1.0,1.25,-1.25
3520,2021-01-17,12:22,152216.0,TRADITIONAL BAGUETTE,-1.0,1.20,-1.20


In [42]:
# Remove negative quantities likely representing returns or corrections
bakery.drop(bakery[bakery["Quantity"] < 0].index, inplace=True)

# Verify
print("Rows:", len(bakery))
print("Negative quantities:", (bakery["Quantity"] < 0).sum())

Rows: 172629
Negative quantities: 0


In [43]:
# Final cleaned dataset
print("Shape:", bakery.shape)
print("Products:", bakery["article"].nunique())
print("Missing values:", bakery.isnull().sum().sum())

display(bakery.head(10))

Shape: (172629, 7)
Products: 26
Missing values: 0


,date,time,ticket_number,article,Quantity,unit_price,revenue
0,2021-01-02,08:38,150040.0,BAGUETTE,1.0,0.90,0.90
1,2021-01-02,08:38,150040.0,PAIN AU CHOCOLAT,3.0,1.20,3.60
2,2021-01-02,09:14,150041.0,PAIN AU CHOCOLAT,2.0,1.20,2.40
4,2021-01-02,09:25,150042.0,TRADITIONAL BAGUETTE,5.0,1.20,6.00
5,2021-01-02,09:25,150043.0,BAGUETTE,2.0,0.90,1.80
6,2021-01-02,09:25,150043.0,CROISSANT,3.0,1.10,3.30
7,2021-01-02,09:27,150044.0,BANETTE,1.0,1.05,1.05
8,2021-01-02,09:32,150045.0,TRADITIONAL BAGUETTE,3.0,1.20,3.60
9,2021-01-02,09:32,150045.0,CROISSANT,6.0,1.10,6.60
10,2021-01-02,09:37,150046.0,PAIN AU CHOCOLAT,6.0,1.20,7.20


In [44]:
# Save cleaned dataset
bakery.to_csv("cleaned_bakery.csv", index=False)